In [0]:
import requests
import json
import time
import os
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor

In [0]:
dbutils.widgets.text("catalog_name", "dbr_dev")
dbutils.widgets.text("schema_name", "weather_bronze")
dbutils.widgets.text("volume_name", "raw")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
volume_name = dbutils.widgets.get("volume_name")

volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/streaming-weather/"

cities_data = {
    "Kraków": {"lat": 50.0647, "lon": 19.9450},
    "Warszawa": {"lat": 52.2297, "lon": 21.0122},
    "Wrocław": {"lat": 51.1079, "lon": 17.0385},
    "Poznań": {"lat": 52.4064, "lon": 16.9252},
    "Gdańsk": {"lat": 54.3520, "lon": 18.6466},
    "Łódź": {"lat": 51.7592, "lon": 19.4560},
    "Szczecin": {"lat": 53.4285, "lon": 14.5528}
}
city_names = list(cities_data.keys())

dbutils.widgets.multiselect("cities", "Kraków", city_names)

selected_cities_str = dbutils.widgets.get("cities")
selected_cities = [city.strip() for city in selected_cities_str.split(",")] if selected_cities_str else []

In [0]:
today = datetime.now().date()
start_date = today - timedelta(days=1)

start_str = start_date.isoformat()
end_str = (start_date + timedelta(days=1)).isoformat()


def save_event_to_volume(city_name, timestamp_str, payload):
    save_ts = timestamp_str.replace(":", "").replace("-", "T")
    file_name = f"weather_{city_name.lower()}_{save_ts}.json"
    file_path = os.path.join(volume_path, file_name)
    
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(payload, f)

In [0]:
def generate_enriched_weather(city_name, lat, lon, start_d, end_d):
    weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&hourly=temperature_2m,wind_speed_10m,wind_direction_10m&start_date={start_d}&end_date={end_d}"
    aq_url = f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={lat}&longitude={lon}&hourly=european_aqi,pm10,pm2_5&start_date={start_d}&end_date={end_d}"    
    
    weather_data = requests.get(weather_url).json().get("hourly", {})
    aq_data = requests.get(aq_url).json().get("hourly", {})        
    
    times = weather_data.get("time", [])
    temps = weather_data.get("temperature_2m", [])
    winds = weather_data.get("wind_speed_10m", [])
    dirs = weather_data.get("wind_direction_10m", [])
    
    aqis = aq_data.get("european_aqi", [])
    pm10s = aq_data.get("pm10", [])
    pm25s = aq_data.get("pm2_5", [])
    
    for i in range(len(times)):
        payload = {
            "city": city_name,
            "temperature": temps[i],
            "wind_speed": winds[i],
            "wind_direction": dirs[i],
            "event_timestamp": times[i],
            "air_quality": aqis[i] if i < len(aqis) else None,
            "pm10": pm10s[i] if i < len(pm10s) else None,
            "pm2_5": pm25s[i] if i < len(pm25s) else None
        }
        save_event_to_volume(city_name, times[i], payload)

In [0]:
def run_historical_ingestion(city_name):
    lat = cities_data[city_name]["lat"]
    lon = cities_data[city_name]["lon"]   
    
    generate_enriched_weather(city_name, lat, lon, start_str, end_str)

if selected_cities:
    with ThreadPoolExecutor(max_workers=len(selected_cities)) as executor:
        for city in selected_cities:
            executor.submit(run_historical_ingestion, city)
            